### 1\使用Chatmodel + ChatPromptTemplate 的实例化方法（另有LLM + PromptTemplate）

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 定义聊天提示词模版
#或使用 ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
        ("human", "你好，最近怎么样？"),
        ("ai", "我很好，谢谢！"),
        ("human", "{user_input}"),
    ]
)
# 格式化聊天提示词模版中的变量
prompt = chat_template.invoke({"name":"小明", "user_input":"你叫什么名字？"})
# 打印格式化后的聊天提示词模版内容
print(prompt)

messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小明。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={})]


### 2\三种方法输出 不同的返回类型 

##### invoke 返回 ChatPromptValue 

In [5]:
prompt = chat_template.invoke({"name":"小明", "user_input":"你叫什么名字？"})
print(type(prompt))
print(prompt)
print(len(prompt.messages))

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小明。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={})]
4


##### format 返回 string 

In [7]:
prompt = chat_template.format(name="小明", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)

<class 'str'>
System: 你是一个有帮助的AI机器人，你的名字是小明。
Human: 你好，最近怎么样？
AI: 我很好，谢谢！
Human: 你能帮我做什么?


##### foemat_massage 返回 消息队列

In [10]:
prompt = chat_template.format_messages(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)

<class 'list'>
[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小谷AI。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


### 3\结合大模型配合Prompt完成实例

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从 .env 文件中加载环境变量
load_dotenv(override=True)
BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")
model = init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=BAILIAN_API_KEY,
    base_url=BAILIAN_API_URL,
)

##提示词模板##

chat_prompt = ChatPromptTemplate(
    [
        ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
        ("human", "你好，最近怎么样？"),
        ("ai", "我很好，谢谢！"),
        ("human", "{user_input}"),
    ]
)

# 输入提示
#prompt_value = chat_prompt.invoke({
#"text":"我今年18岁，我的舅舅今年38岁，我的爷爷今年72岁，我和舅舅一共多少岁了？"
#})
# 输入提示
prompt_value = chat_prompt.format(
name = "小明",
user_input="我今年18岁，我的舅舅今年38岁，我的爷爷今年72岁，我和舅舅一共多少岁了？"
)
# 输入提示
#prompt_value = chat_prompt.format_messages(
#name = "小明",
#user_input = "我今年18岁，我的舅舅今年38岁，我的爷爷今年72岁，我和舅舅一共多少岁了？"
#)

######结合提示词，调用大模型#########
# 得到模型的输出
output = model.invoke(prompt_value)
# 打印输出内容
print(output.content)


你和舅舅一共是：

18 + 38 = 56 岁。


### 4\丰富初始化参数类型

###### ***参数是列表类型，列表的元素可以是字符串、字典、字符串构成的元组、消息类型、提示词模板类型、消息提示词模板类型等***


##### 综合使用（下面拆解）

In [ ]:
from langchain_core.prompts import (
ChatPromptTemplate,
SystemMessagePromptTemplate,
HumanMessagePromptTemplate,
)
from langchain_core.messages import SystemMessage, HumanMessage

# 示例 1: 使用 BaseMessage（已实例化的消息）
system_msg = SystemMessage(content="你是一个AI工程师。")
human_msg = HumanMessage(content="你好！")

# 示例 2: 使用 BaseMessagePromptTemplate
system_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}.")
human_prompt = HumanMessagePromptTemplate.from_template("{user_input}")

# 示例 3: 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt = ChatPromptTemplate.from_messages([("system", "嵌套提示词")])

prompt = ChatPromptTemplate.from_messages([
    system_msg, # MessageLike (BaseMessage)
    human_msg, # MessageLike (BaseMessage)
    system_prompt, # MessageLike (BaseMessagePromptTemplate)
    human_prompt, # MessageLike (BaseMessagePromptTemplate)
    nested_prompt, # MessageLike (BaseChatPromptTemplate)
])
prompt.invoke({"role":"人工智能专家","user_input":"介绍一下大模型的应用场景"})


In [ ]:
#str
chat_template = ChatPromptTemplate.from_messages(
[
"Hello, {name}!" # 等价于 ("human", "Hello, {name}!")
]
)

In [ ]:
#tuple
prompt = ChatPromptTemplate.from_messages([
("system", "你的名字是{role}."),
("human", "很高兴认识你"),
])

In [ ]:
#dict
prompt = ChatPromptTemplate.from_messages([
{"role": "system", "content": "你的名字是{role}."},
{"role": "human", "content":"很高兴认识你"},
])

In [ ]:
#XxxMessage
chat_prompt_template = ChatPromptTemplate.from_messages([
SystemMessage(content="我是一个贴心的智能助手"),
HumanMessage(content="我的问题是:人工智能英文怎么说？")
])

**注意：在XxxMessage中不能有占位符。**

In [ ]:
# MessagePromptTemplate列表类型
# 创建消息模板
system_message_prompt = SystemMessagePromptTemplate.from_template("你是一个
{role}")
human_message_prompt = HumanMessagePromptTemplate.from_template("给我解释
{concept}，用浅显易懂的语言")
# 组合成聊天提示模板
chat_prompt = ChatPromptTemplate.from_messages([
system_message_prompt,
human_message_prompt
])

In [ ]:
# BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt_template1 = ChatPromptTemplate.from_messages([
("system", "我是一个人工智能助手，我的名字叫{name}")
])
nested_prompt_template2 = ChatPromptTemplate.from_messages([
("human", "很高兴认识你,我的问题是{question}")
])
prompt_template = ChatPromptTemplate.from_messages([
nested_prompt_template1,nested_prompt_template2
])